[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Query Parameters


## What you will be able to do

Send any value in a query or a path and have the server receive exactly that value: text with
spaces, `&`, `+` or `ø` in it, a list, a date, and a parameter left out when it has no value. Build
a query by hand when there is no `params` to do it for you.


## The idea

### The problem

A query is text, and some of its characters have jobs. `&` separates one parameter from the next,
`=` separates a name from its value, `#` ends the part of a URL that a client sends, and a `+` in a
query stands for a space. A value that contains one of them, written straight into a URL, changes
the request instead of traveling inside it.

Nothing reports it. A search for `Bergen & Oslo` written into an f-string reaches the server as a
search for `Bergen ` and a second parameter named ` Oslo`. A time with its time zone, `06:00+01:00`,
arrives as `06:00 01:00`. A note that contains `#` loses the rest of itself, and every parameter
after it. The server answers `200 OK` each time, to a question nobody asked. A path has the same
weakness: a station id taken from user input as `oslo/../svalbard` fetches Svalbard.

Code tested with values like `tromso` and `2025-01-15` never meets any of this, which is why it
works until real data arrives.

### What a query string is

> A **query string** is the part of a URL after `?` and before any `#`: a list of **query
> parameters**, each written `name=value` and separated by `&`. Names and values are
> **percent-encoded**: a character that a URL cannot contain, or that would be read as part of its
> structure, is written as `%` and two hexadecimal digits for each byte of its UTF-8 encoding. A
> space becomes `%20`, or `+` in a query, and `ø` becomes `%C3%B8`.

### Why it works that way

- **A URL has a small alphabet.** Letters, digits and a few symbols travel as themselves. Anything
  else, from a space to `ø`, has to be encoded, and so does any symbol that would otherwise act as
  structure.
- **Encoding happens once, where the URL is built.** The server decodes every name and value
  exactly once. A value encoded twice arrives still encoded, and a value not encoded at all has its
  symbols read as structure.
- **A query has no types.** Every value arrives as text. A number, a list, a date or a flag is a
  convention between a client and an API, written down in the API's documentation: a list may be a
  repeated name or a comma-separated value, and a flag may be `true` or `1`.
- **A space has two encodings.** HTML forms send a space as `+`, so servers read `+` in a query as a
  space, and a `+` that belongs to a value has to travel as `%2B`. requests and Python's `urlencode`
  follow the form convention.
- **A path is not a query.** In a path, `/` separates segments and `..` means the segment above,
  which HTTP clients resolve before sending. requests encodes the values in `params` completely, but
  in the rest of a URL it encodes only the characters a URL cannot contain, so a value placed in a
  path needs encoding by hand.

### Where you will meet this

Every search box puts its words in a query, as `?q=` in a web search. Open-Meteo's documentation
page builds a query as you tick options, and takes several daily variables as one comma-separated
value. GitHub's search API takes a whole search language in one parameter, as in
`q=language:python stars:>1000`, where the `:`, the `>` and the space all need encoding. The
**Pagination** notebook sends `page` and `per_page` parameters, the **Authentication** notebook
explains why a key sent in a query ends up in server logs, and FastAPI, in the **Your First API
Server** notebook, turns query parameters into typed function arguments on the server.

### What this notebook covers

- `params`, and the practice API's `/echo`, which shows what the server received
- The characters that need encoding, and what each becomes
- A space, as `+` or `%20`
- Lists, as a repeated name or as one comma-separated value, on `/echo` and on Open-Meteo
- Numbers, dates and flags, and `None` for a parameter left out
- A URL that already has a query
- `urlencode` for a query built by hand, and `quote` for a path parameter
- A weather table for every station, from a query built entirely from Python values
- Five errors, from a query written into an f-string to a path value that fetched a different
  station

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

response = requests.get("http://127.0.0.1:8765/echo", timeout=10,
                        params={"q": "Bergen & Oslo", "near": "Tromsø"})

print(response.url)
print(response.json()["args"])
```

```
http://127.0.0.1:8765/echo?q=Bergen+%26+Oslo&near=Troms%C3%B8
{'q': ['Bergen & Oslo'], 'near': ['Tromsø']}
```

requests encoded both values on the way out, and `/echo`, an address on the practice API that
responds with the query it received, shows that the server decoded them back into exactly what was
sent.


## Setup

Eight imports, the last of them the practice API.

- `requests` sends every request in this notebook
- `quote` and `urlencode`, from `urllib.parse`, encode a path parameter and a query by hand
- `date` and `datetime` make the dates and times that a query sends as text
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `open_meteo()` returns Open-Meteo's address, or the address of the practice API's recording of
  it when Open-Meteo is not answering

If Open-Meteo stops answering while you work through the notebook, run this cell again: it checks
again, and the Open-Meteo cells switch to the recording.


In [1]:
import importlib
import sys
import urllib.request
from datetime import date, datetime
from pathlib import Path
from urllib.parse import quote, urlencode

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("The practice API is running at", BASE)
print("Open-Meteo's archive is at", OPEN_METEO)


The practice API is running at http://127.0.0.1:8765
Open-Meteo's archive is at https://archive-api.open-meteo.com/v1/archive


## Worked examples

### A query from a dictionary, and what the server received

The practice API has an endpoint for this notebook. `/echo` responds with the query its request
arrived with: `query` is the query string exactly as it was sent, still encoded, and `args` is the
same string decoded by the server, name by name, with the `parse_qs` that the **What an API Is**
notebook used. Public testing services such as httpbin offer the same kind of endpoint.


In [2]:
response = requests.get(f"{BASE}/echo", params={"station": "tromso", "days": 3}, timeout=10)

print("sent: ", response.url)
print("query:", response.json()["query"])
print("args: ", response.json()["args"])


sent:  http://127.0.0.1:8765/echo?station=tromso&days=3
query: station=tromso&days=3
args:  {'station': ['tromso'], 'days': ['3']}


requests wrote the dictionary into the URL after a `?`, in the order it was given, as `name=value`
pairs joined by `&`. The number `3` traveled as the text `3`, because a query holds only text. On
the server every name maps to a list, since a name may appear more than once.

### Characters that need encoding

Here is a value containing each character that has a job in a URL, and three that a URL cannot
contain at all. requests encodes every value in `params`, and `/echo` shows how each one traveled and
what arrived:


In [3]:
values = {"space": "Bergen Oslo", "ampersand": "Bergen & Oslo", "equals": "a=b",
          "plus": "06:00+01:00", "hash": "sensor #2", "slash": "oslo/bergen", "question": "why?",
          "percent": "50%", "o_slash": "Tromsø", "degree": "°C"}

echo = requests.get(f"{BASE}/echo", params=values, timeout=10).json()
sent = dict(pair.split("=", 1) for pair in echo["query"].split("&"))

for name, value in values.items():
    print(f"{value!r:<17} sent as {sent[name]:<20} arrived as {echo['args'][name][0]!r}")


'Bergen Oslo'     sent as Bergen+Oslo          arrived as 'Bergen Oslo'
'Bergen & Oslo'   sent as Bergen+%26+Oslo      arrived as 'Bergen & Oslo'
'a=b'             sent as a%3Db                arrived as 'a=b'
'06:00+01:00'     sent as 06%3A00%2B01%3A00    arrived as '06:00+01:00'
'sensor #2'       sent as sensor+%232          arrived as 'sensor #2'
'oslo/bergen'     sent as oslo%2Fbergen        arrived as 'oslo/bergen'
'why?'            sent as why%3F               arrived as 'why?'
'50%'             sent as 50%25                arrived as '50%'
'Tromsø'          sent as Troms%C3%B8          arrived as 'Tromsø'
'°C'              sent as %C2%B0C              arrived as '°C'


Every value arrived as it was sent. Each character that could have been read as structure traveled
as `%` and two hexadecimal digits: `&` as `%26`, `=` as `%3D`, `+` as `%2B`, `#` as `%23`, and `:`
as `%3A`, though it has no job in a query. `ø` is two bytes in UTF-8, so it traveled as two escapes,
and `%` itself traveled as `%25`, so the server could tell it from the start of an escape.

The cell splits the encoded query at `&` and `=` to show each value as it was sent. That is safe only
because every value was encoded first: an encoded value contains neither character.

### A space, as + or %20

requests wrote each space as `+`. `quote`, and a browser's address bar, write `%20` instead. Here
are both, and an encoded plus, typed into queries by hand:


In [4]:
for query in ["q=Bergen+Oslo", "q=Bergen%20Oslo", "q=06:00%2B01:00"]:
    received = requests.get(f"{BASE}/echo?{query}", timeout=10).json()["args"]["q"][0]
    print(f"{query:<17} arrived as {received!r}")


q=Bergen+Oslo     arrived as 'Bergen Oslo'
q=Bergen%20Oslo   arrived as 'Bergen Oslo'
q=06:00%2B01:00   arrived as '06:00+01:00'


Both spellings of a space arrived as a space, and `%2B` arrived as a plus. HTML forms introduced
`+` for a space, and servers still read it that way in a query, which is why a `+` inside a value
has to travel as `%2B`. In a path, a `+` is only a `+`.

### Lists: a repeated name, or one value with commas

A query has no list type, so each API chooses a convention, and its documentation says which.
requests produces either. A list sends the name once for each item, and a string joined with
commas sends it once:


In [5]:
variables = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"]

for params in [{"daily": variables}, {"daily": ",".join(variables)}]:
    echo = requests.get(f"{BASE}/echo", params=params, timeout=10).json()
    print(echo["query"])
    print("   ", echo["args"])


daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum
    {'daily': ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']}
daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum
    {'daily': ['temperature_2m_max,temperature_2m_min,precipitation_sum']}


The first arrived as three values for one name. The second arrived as a single value, with its
commas encoded on the way and decoded again, for the server to split itself. Which of the two an API
reads is up to the API. Open-Meteo's documentation asks for commas, so here are three daily
variables for Tromso in a single request:


In [6]:
response = requests.get(OPEN_METEO, timeout=30, params={
    "latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15", "end_date": "2025-01-17",
    "daily": ",".join(variables), "models": "era5"})

daily, units = response.json()["daily"], response.json()["daily_units"]
for variable in variables:
    print(f"{variable:<19} {units[variable]:<3} {daily[variable]}")


temperature_2m_max  °C  [6.5, 7.1, 8.6]
temperature_2m_min  °C  [2.7, 5.8, 6.2]
precipitation_sum   mm  [23.7, 21.1, 18.8]


One parameter carried all three variables, and the response holds a list of values for each, with
its unit under `daily_units`.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.

### Numbers, dates and flags

requests turns every value in `params` into text with `str`. Here is what that makes of five values
that are not text:


In [7]:
params = {"latitude": 69.65, "days": 3, "start": date(2025, 1, 15),
          "time": datetime(2025, 1, 15, 6, 30), "hourly": True}

echo = requests.get(f"{BASE}/echo", params=params, timeout=10).json()
for name, value in params.items():
    print(f"{name:<9} {value!r:<38} arrived as {echo['args'][name][0]!r}")


latitude  69.65                                  arrived as '69.65'
days      3                                      arrived as '3'
start     datetime.date(2025, 1, 15)             arrived as '2025-01-15'
time      datetime.datetime(2025, 1, 15, 6, 30)  arrived as '2025-01-15 06:30:00'
hourly    True                                   arrived as 'True'


For the numbers and the `date`, `str` gives the text an API expects: `2025-01-15` is the ISO 8601
form that Open-Meteo's documentation asks for, as most APIs' documentation does. The `datetime`
arrived with a space where ISO 8601 puts a `T`, and `True` arrived as `True`, where an API that
takes a flag usually documents `true`. When `str` does not give the documented text, convert the
value yourself, with `isoformat()` for dates and times:


In [8]:
params = {"start": date(2025, 1, 15).isoformat(), "time": datetime(2025, 1, 15, 6, 30).isoformat(),
          "hourly": "true"}

print(requests.get(f"{BASE}/echo", params=params, timeout=10).json()["args"])


{'start': ['2025-01-15'], 'time': ['2025-01-15T06:30:00'], 'hourly': ['true']}


### A parameter left out: None

An optional parameter is one a program sends only sometimes, such as Open-Meteo's
`temperature_unit`, which means Celsius unless it is sent. requests leaves out any parameter whose
value is `None`, so a single function covers both cases:


In [9]:
def station_weather(latitude, longitude, unit=None):
    """Three daily mean temperatures for a place, from Open-Meteo, in Celsius unless a unit is given."""
    return requests.get(OPEN_METEO, timeout=30, params={
        "latitude": latitude, "longitude": longitude,
        "start_date": date(2025, 1, 15).isoformat(), "end_date": date(2025, 1, 17).isoformat(),
        "daily": "temperature_2m_mean", "models": "era5", "temperature_unit": unit})


for unit in [None, "fahrenheit"]:
    response = station_weather(60.39, 5.32, unit)
    print(response.url.removeprefix(OPEN_METEO))
    print("   ", response.json()["daily_units"]["temperature_2m_mean"], response.json()["daily"]["temperature_2m_mean"])


?latitude=60.39&longitude=5.32&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5
    °C [7.8, 8.0, 8.0]
?latitude=60.39&longitude=5.32&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5&temperature_unit=fahrenheit
    °F [46.0, 46.3, 46.4]


With `unit` as `None`, the query has no `temperature_unit` at all, and Open-Meteo used its default.
With `"fahrenheit"`, the parameter appears at the end, where the dictionary put it. The same
function asks for Bergen both ways, without an `if` around the parameter.

### A URL that already has a query

`params` adds to whatever query the URL already has. That helps with a URL an API hands you to call
next, and causes trouble when a name ends up in both places:


In [10]:
response = requests.get(f"{BASE}/echo?models=era5", timeout=10,
                        params={"daily": "temperature_2m_mean", "models": "best_match"})

print(response.json()["query"])
print(response.json()["args"])


models=era5&daily=temperature_2m_mean&models=best_match
{'models': ['era5', 'best_match'], 'daily': ['temperature_2m_mean']}


The server received two values for `models`. APIs disagree about which of two values counts: some
read the first, some the last, and some reject the request, so keep each parameter in one place. A
URL that arrives with its query already built, such as the address of a next page, is the usual
source, and the **Pagination** notebook follows addresses like that.

### A query built by hand: urlencode

Some code needs the URL itself rather than a response: a link to print, a curl command to show, or
a request through `urllib.request`, which has no `params`. `urlencode` builds a query string the
way requests does:


In [11]:
params = {"q": "Bergen & Oslo", "id": ["oslo", "bergen"]}

print(urlencode(params, doseq=True))
print(urlencode(params, doseq=True, quote_via=quote))


q=Bergen+%26+Oslo&id=oslo&id=bergen
q=Bergen%20%26%20Oslo&id=oslo&id=bergen


`doseq=True` writes a list as a repeated name, as requests does, and Common errors shows what
happens without it. `quote_via=quote` writes a space as `%20` instead of `+`. Either string goes
after a `?`, and the server receives the same values:


In [12]:
url = f"{BASE}/echo?{urlencode(params, doseq=True, quote_via=quote)}"

print(url)
print(requests.get(url, timeout=10).json()["args"])


http://127.0.0.1:8765/echo?q=Bergen%20%26%20Oslo&id=oslo&id=bergen
{'q': ['Bergen & Oslo'], 'id': ['oslo', 'bergen']}


### A path parameter, encoded with quote

A value that goes into a path, such as a station id, is not encoded for you. `quote` encodes it,
and `safe=""` tells it to encode `/` as well, so the value stays a single segment of the path:


In [13]:
for station_id in ["tromso", "north cape", "oslo/bergen"]:
    response = requests.get(f"{BASE}/stations/{quote(station_id, safe='')}", timeout=10)
    body = response.json()
    print(f"{station_id!r:<14} {response.url.removeprefix(BASE):<25} {response.status_code}",
          body.get("name") or body["error"])


'tromso'       /stations/tromso          200 Tromso
'north cape'   /stations/north%20cape    404 no station with id 'north cape'
'oslo/bergen'  /stations/oslo%2Fbergen   404 no station with id 'oslo/bergen'


The practice API decodes a path parameter before looking it up, as real servers do, so its `404`s
name the two ids it does not have exactly as they were written, `/` included. `quote` leaves `/`
unencoded unless told otherwise, because it was written for whole paths, where `/` belongs; Common
errors shows what that default does to an id like `oslo/../svalbard`.

### A weather table for every station

Everything in this notebook, in one function. `daily_weather` asks Open-Meteo for any list of daily
variables at a station, over any dates, and builds the query entirely from Python values: the
coordinates from the practice API, the dates from `date`, the variables joined with commas, and a
unit left out unless one is given.


In [14]:
def daily_weather(station, variables, start, end, unit=None):
    """Each daily variable at a station, from Open-Meteo, as {variable: (unit, values)}."""
    response = requests.get(OPEN_METEO, timeout=30, params={
        "latitude": station["latitude"],
        "longitude": station["longitude"],
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": ",".join(variables),
        "models": "era5",
        "temperature_unit": unit,
    })
    response.raise_for_status()
    body = response.json()
    return {variable: (body["daily_units"][variable], body["daily"][variable]) for variable in variables}


variables = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"]

for summary in requests.get(f"{BASE}/stations", timeout=10).json():
    station = requests.get(f"{BASE}/stations/{quote(summary['id'], safe='')}", timeout=10).json()
    print(station["name"])
    for variable, (unit, values) in daily_weather(station, variables, date(2025, 1, 15), date(2025, 1, 17)).items():
        print(f"    {variable:<19} {unit:<3} {values}")


Bergen
    temperature_2m_max  °C  [8.3, 8.2, 8.9]
    temperature_2m_min  °C  [6.9, 7.6, 6.4]
    precipitation_sum   mm  [13.2, 5.6, 4.0]
Oslo
    temperature_2m_max  °C  [2.8, 3.2, 4.8]
    temperature_2m_min  °C  [-3.6, 1.3, 1.4]
    precipitation_sum   mm  [0.1, 0.0, 0.3]
Svalbard
    temperature_2m_max  °C  [-11.0, -12.3, -12.1]
    temperature_2m_min  °C  [-13.6, -13.1, -15.1]
    precipitation_sum   mm  [0.4, 5.7, 4.7]
Tromso
    temperature_2m_max  °C  [6.5, 7.1, 8.6]
    temperature_2m_min  °C  [2.7, 5.8, 6.2]
    precipitation_sum   mm  [23.7, 21.1, 18.8]


Twelve rows from four requests to Open-Meteo, with every value in them encoded by requests or by
`quote`.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.

### Where each part came from

| In the program | What it relies on | The section that showed it |
|---|---|---|
| `params={...}` | a query built from a dictionary, every value encoded | A query from a dictionary, and what the server received |
| `start.isoformat()` | a date sent as the text the API documents | Numbers, dates and flags |
| `",".join(variables)` | a list sent as one comma-separated value, as Open-Meteo documents | Lists: a repeated name, or one value with commas |
| `"temperature_unit": unit` | `None` leaves a parameter out | A parameter left out: None |
| `quote(summary['id'], safe='')` | a path parameter kept to one segment | A path parameter, encoded with quote |
| `response.raise_for_status()` | an error status stops the program | the **Your First Request** notebook |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/05-query-parameters-solutions.ipynb).

**1.** Send `/echo` a parameter `station` set to `Tromsø` and a parameter `note` set to
`cold & dark`. Print the URL requests sent and the `args` the server received.


In [15]:
# your code here


**2.** Use `urlencode` to build a query in which `q` is `50% chance of snow`, once with each space
as `+` and once as `%20`, and print both.


In [16]:
# your code here


**3.** Send `/echo` the station ids `oslo`, `bergen` and `tromso`, first as a repeated `id`
parameter and then as one comma-separated `ids` parameter. Print the `args` the server received
each time.


In [17]:
# your code here


**4.** Send `/echo` a parameter `python` set to the Python value `True`, and a parameter
`documented` set to the string `"true"`. Print the query the server received.


In [18]:
# your code here


**5.** Use `station_weather`, from the section on leaving a parameter out, to ask Open-Meteo for
Tromso's daily mean temperatures in Fahrenheit. Tromso is at latitude `69.65` and longitude `18.96`.
Print the URL requests sent and the values.


In [19]:
# your code here


**6.** Request the station whose id is `svalbard?`, with the question mark, twice: once with the id
encoded by `quote` with `safe=""`, and once written into the path as it is. For each, print the
status code and the station's name or the error.


In [20]:
# your code here


## Common errors

### No error, and the wrong values: a query written into an f-string


In [21]:
response = requests.get(f"{BASE}/echo?q=Bergen & Oslo&time=06:00+01:00&note=sensor #2&days=3", timeout=10)
print(response.json()["args"])


{'q': ['Bergen '], ' Oslo': [''], 'time': ['06:00 01:00'], 'note': ['sensor ']}


Four parameters were written, and the server received something different for every one of them.
The `&` in `Bergen & Oslo` ended the value and began a parameter named ` Oslo`. The `+` of the time
zone offset arrived as a space. `#` began the URL's fragment, which a client never sends, so `2` and
`days=3` were not sent at all. Give the values to `params`, and requests encodes each one:


In [22]:
response = requests.get(f"{BASE}/echo", timeout=10, params={
    "q": "Bergen & Oslo", "time": "06:00+01:00", "note": "sensor #2", "days": 3})
print(response.json()["args"])


{'q': ['Bergen & Oslo'], 'time': ['06:00+01:00'], 'note': ['sensor #2'], 'days': ['3']}


### No error, and %20 inside the value: a value encoded twice


In [23]:
response = requests.get(f"{BASE}/echo", params={"q": "Bergen%20Oslo"}, timeout=10)
print(response.json()["query"], "->", response.json()["args"])


q=Bergen%2520Oslo -> {'q': ['Bergen%20Oslo']}


The value was already encoded, as a value copied from a browser's address bar often is, and requests
encoded it again: its `%` traveled as `%25`. The server decoded once, as servers do, and received
the encoding instead of the value. Give `params` the value itself:


In [24]:
response = requests.get(f"{BASE}/echo", params={"q": "Bergen Oslo"}, timeout=10)
print(response.json()["query"], "->", response.json()["args"])


q=Bergen+Oslo -> {'q': ['Bergen Oslo']}


### No error, and a list's brackets inside the value: urlencode without doseq


In [25]:
query = urlencode({"id": ["oslo", "bergen"]})
print(query)
print(requests.get(f"{BASE}/echo?{query}", timeout=10).json()["args"])


id=%5B%27oslo%27%2C+%27bergen%27%5D
{'id': ["['oslo', 'bergen']"]}


Without `doseq=True`, `urlencode` turns every value into text with `str`, and the text of a list is
its brackets, quotes and commas, all encoded. The server received a single value: the text of a
Python list. With `doseq=True`, each item becomes a value of its own:


In [26]:
query = urlencode({"id": ["oslo", "bergen"]}, doseq=True)
print(query)
print(requests.get(f"{BASE}/echo?{query}", timeout=10).json()["args"])


id=oslo&id=bergen
{'id': ['oslo', 'bergen']}


### No error, and a different station: a path value quote did not fully encode


In [27]:
station_id = "oslo/../svalbard"
response = requests.get(f"{BASE}/stations/{quote(station_id)}", timeout=10)
print(response.url.removeprefix(BASE), response.status_code, response.json()["name"])


/stations/svalbard 200 Svalbard


`quote` leaves `/` unencoded by default, so the id became three segments of the path, `oslo`, `..`
and `svalbard`, and requests resolved `..` before sending, the way a file path would be resolved.
The request asked for Svalbard, and got it, with a `200`. When an id comes from user input, the same
mistake lets the user reach any path on the API. With `safe=""`, the id stays one segment:


In [28]:
response = requests.get(f"{BASE}/stations/{quote(station_id, safe='')}", timeout=10)
print(response.url.removeprefix(BASE), response.status_code, response.json()["error"])


/stations/oslo%2F..%2Fsvalbard 404 no station with id 'oslo/../svalbard'


### TypeError: quote_from_bytes() expected bytes


In [29]:
quote(69.65, safe="")


TypeError: quote_from_bytes() expected bytes

`quote` encodes text, and `69.65` is a number, so it raises before anything is sent. requests calls
`str` on every value in `params`, and `quote` does not. Convert the value first:


In [30]:
print(quote(str(69.65), safe=""))


69.65


## Recap

- A query is text: `name=value` pairs after `?`, joined by `&`, with every name and value
  percent-encoded.
- Give values to `params` and let requests encode them, rather than writing them into a URL.
- A space travels as `+` or `%20`, a real `+` as `%2B`, and a value encoded twice arrives still
  encoded.
- A list travels as a repeated name or as one comma-separated value, whichever the API documents,
  and `urlencode` needs `doseq=True` for the first.
- requests sends numbers and dates as their usual text and `True` as `True`, and leaves out a
  parameter whose value is `None`.
- Encode a path parameter with `quote(value, safe="")`, so that `/`, `?`, `#` and `..` stay inside
  the value.


## What is next

The **JSON in a Response** notebook. The weather table here reached into Open-Meteo's response by
name, because the shape of the response was known in advance; that notebook reads JSON whose shape
is not, finding the field you want inside nested objects and lists.


---

&#8592; **Previous:** [Status Codes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/04-status-codes.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [JSON in a Response](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/06-json-in-a-response.ipynb) &#8594;
